# Урок 8.2. Окружение и компоненты: конфиг, чанкинг, индексация

Проект начинается не с эндпоинтов, а с фундамента: конфигурация, структура,
подготовка данных. В этом уроке пройдём путь документа от `.md`-файла до точки
в Qdrant — и запустим настоящую индексацию.

🎯 **Цели урока:**

- Конфигурация через pydantic-settings: приоритеты источников, валидация, `.env`.
- Структура проекта: где граница между роутерами, сервисами и скриптами.
- Чанкинг markdown: двухуровневая стратегия «секции → абзацы с перекрытием».
- Идемпотентная индексация: uuid5 вместо автоинкремента (живой запуск).

**Что понадобится:** запущенный Qdrant (`docker compose up -d` в модуле 2 или
в `project/`), пакеты из `project/requirements.txt`.

## 1. Конфигурация: pydantic-settings

Всё, что может отличаться между машинами и окружениями, живёт в
[`app/config.py`](project/app/config.py) — классе `Settings` на pydantic-settings.
Три свойства делают его лучше самодельного `os.getenv` (которым мы жили
с урока 1.4):

1. **Приоритеты источников**: аргументы конструктора → переменные окружения
   (`APP_*`) → файл `.env` → значения по умолчанию. Одна строчка в тестах
   (`Settings(environment="test")`) перекрывает всё.
2. **Валидация на старте**: `search_top_k: int = Field(ge=1, le=20)` — опечатка
   в переменной окружения уронит сервис при запуске с внятной ошибкой, а не
   в полночь на проде странным поведением.
3. **Документированность**: класс — это и есть перечень всех настроек; никакой
   охоты за `os.getenv` по кодовой базе.

Проверим приоритеты вживую:

In [ ]:
import os
import sys
from pathlib import Path

sys.path.insert(0, str(Path("project").resolve()))   # импортируем app.* из project/

from pydantic import ValidationError

from app.config import Settings

# 1. дефолты
settings = Settings(_env_file=None)
print("дефолт:", settings.qdrant_url, "|", settings.llm_model, "| top_k =", settings.search_top_k)

# 2. переменная окружения перекрывает дефолт
os.environ["APP_SEARCH_TOP_K"] = "7"
print("env:   top_k =", Settings(_env_file=None).search_top_k)

# 3. аргумент конструктора перекрывает окружение
print("kwargs: top_k =", Settings(_env_file=None, search_top_k=3).search_top_k)

# 4. мусор в окружении = падение на старте, а не сюрприз в рантайме
os.environ["APP_SEARCH_TOP_K"] = "сто"
try:
    Settings(_env_file=None)
except ValidationError as exc:
    print("мусор:", str(exc).splitlines()[1].strip())
finally:
    del os.environ["APP_SEARCH_TOP_K"]

Обратите внимание на `127.0.0.1` в дефолтах вместо `localhost` — это не
суеверие: на Windows `localhost` сперва резолвится в `::1`, и httpx теряет
~2 секунды на каждом свежем соединении (мы поймали это хронометражем в уроке
3.2 и с тех пор пишем адрес цифрами).

## 2. Структура: слои и их обязанности

```
project/
├── app/
│   ├── config.py        ← настройки (вы здесь)
│   ├── schemas.py       ← Pydantic-контракты API (граница с клиентом)
│   ├── routers/         ← HTTP: парсинг, коды ответов, БЕЗ бизнес-логики
│   ├── services/        ← логика: rag, embeddings, vectorstore, llm, history
│   ├── dependencies.py  ← мост роутеры→сервисы через app.state (DI)
│   ├── middleware.py    ← request-id, access-лог, метрики
│   └── main.py          ← фабрика create_app() + lifespan
├── scripts/index_docs.py ← индексация: отдельный процесс, не эндпоинт!
├── data/docs/            ← знания
└── tests/                ← юнит + интеграционные на фейках
```

Два решения, о которые чаще всего спотыкаются:

- **Индексация — скрипт, а не эндпоинт.** Переиндексация — редкая
  административная операция с правами на запись в базу знаний. Делать её
  HTTP-ручкой = добавить в сервис самый опасный эндпоинт и вечный вопрос
  «кто может его дёргать». Скрипт запускает человек или CI.
- **DI через `app.state`, а не глобали.** Сервисы создаются один раз в lifespan
  и складываются в состояние приложения; роутеры получают их через `Depends`.
  В тестах в state кладутся фейки — без importlib-магии и monkeypatching
  (это окупится в уроке 8.5).

## 3. Чанкинг: секции, затем абзацы

Модуль 2 научил: как порежешь — так и найдёшь. Для **markdown со структурой**
у нас стратегия лучше, чем «абзацы с склейкой» из kb-search
([`app/services/chunking.py`](project/app/services/chunking.py)):

1. Документ режется на секции по `## `-заголовкам — автор документации уже
   разметил смысловые блоки, грех не воспользоваться.
2. Секции длиннее `chunk_size` дорезаются по абзацам с **перекрытием**: мысль,
   разрезанная на границе, целиком попадает хотя бы в один чанк.

Посмотрим на настоящем документе:

In [ ]:
from app.services.chunking import split_markdown

deploy_md = Path("project/data/docs/05-deploy-guide.md").read_text(encoding="utf-8")
chunks = split_markdown(deploy_md, source="05-deploy-guide.md",
                        chunk_size=1000, chunk_overlap=200)

for chunk in chunks:
    first_line = chunk.text.splitlines()[0]
    print(f"#{chunk.chunk_index} [{len(chunk.text):>4} симв.] {first_line}")

print("\n--- чанк #2 целиком ---")
print(chunks[2].text)

Каждая секция гида по деплою стала отдельным чанком — и это идеальный размер
для поиска: чанк #2 «Деплой на production» содержит и тег `vX.Y.Z`, и job
`deploy-prod`, и условия — самодостаточный ответ на вопрос «как деплоить на
прод». Заголовок секции остаётся в тексте чанка — это те же «контекстные
заголовки» из урока 2.6, только бесплатно: их написал автор документа.

## 4. Идемпотентная индексация

Главная деталь [`scripts/index_docs.py`](project/scripts/index_docs.py) —
детерминированные ID точек (урок 2.4 всё ещё с нами):

```python
def point_id_for(source: str, chunk_index: int) -> str:
    return str(uuid.uuid5(uuid.NAMESPACE_URL, f"{source}#{chunk_index}"))
```

Повторная индексация того же файла порождает **те же UUID** — upsert обновляет
существующие точки вместо создания дублей. Скрипт можно дёргать после каждой
правки документации, не думая о состоянии базы. (Хвост-ловушку — что будет,
если документ стал КОРОЧЕ, — разберём в квизе.)

Запустим по-настоящему:

In [ ]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "-X", "utf8", "scripts/index_docs.py", "--recreate"],
    cwd="project", capture_output=True, text=True, encoding="utf-8", timeout=600,
)
tail = [line for line in result.stderr.splitlines() + result.stdout.splitlines()
        if "| INFO" in line and ("чанков" in line or "точек" in line or "Эмбеддинги" in line)]
print("\n".join(tail[-6:]))
print("exit code:", result.returncode)

Мой прогон: **12 документов → 61 чанк, эмбеддинги за 1.5 секунды, 61 точка
в коллекции** — вся индексация с пересозданием укладывается в полминуты (большая
часть которой — загрузка модели e5 в память; сами вектора для 61 чанка почти
бесплатны).

Флаг `--recreate` мы использовали сознательно: коллекция пересоздана с нуля,
и число точек равно числу чанков по построению. Без флага скрипт идемпотентно
обновил бы существующие точки — быстрый путь для правок документации
(а его хвост-ловушку разберём в квизе).

## ⚠️ Частые ошибки

1. **Настройки читаются при импорте модуля.** `SETTINGS = Settings()` на уровне
   модуля = нельзя переопределить в тестах. У нас — `get_settings()` c
   `lru_cache` и фабрика `create_app(settings)`.
2. **Индексация эндпоинтом.** Самый опасный эндпоинт сервиса + вопросы
   авторизации бесплатно. Скрипт/CI-job.
3. **Автоинкрементные ID точек.** Каждая переиндексация плодит дубли, поиск
   начинает возвращать три копии одного чанка. Детерминированный uuid5.
4. **Чанк без заголовка секции.** Режете по `##`, а заголовок выбрасываете —
   теряете контекстный заголовок, который у markdown уже есть.
5. **`.env` в git.** В репозитории живёт только `.env.example` с плейсхолдерами;
   сам `.env` — в `.gitignore` (боевые адреса и будущие ключи — не для истории
   коммитов).

## Мини-квиз

**1. Документ был на 7 чанков, после правки стал на 5. Что останется в Qdrant после переиндексации без `--recreate`?**

<details><summary>Ответ</summary>

7 точек: 5 обновлённых и 2 «сироты» с хвостов старой нарезки (upsert по uuid5
не удаляет лишнее). Лечения два: `--recreate` (пересоздать коллекцию — наш
дефолт для маленькой базы) или удалять точки источника перед upsert'ом.
Мы наступали на эти грабли в модуле 2 — потому в kb-search коллекция
пересоздаётся.
</details>

**2. Чем плох `chunk_size=200` для нашей документации? А `chunk_size=8000`?**

<details><summary>Ответ</summary>

200 — секции раскрошатся на обрывки без контекста (и «Откат» потеряет строку
про миграции); 8000 — весь документ станет одним чанком, вектор «обо всём»
размывается (урок 2.5), а в промпт генерации поедут тысячи лишних токенов.
1000 с перекрытием 200 подобран под размер секций наших доков.
</details>

**3. Почему `Settings(environment="test")` в тестах лучше, чем `os.environ["APP_ENVIRONMENT"]="test"`?**

<details><summary>Ответ</summary>

Аргумент конструктора имеет высший приоритет и действует только на этот
экземпляр — без глобального состояния, которое протекает между тестами
и требует уборки. Плюс `lru_cache` на get_settings() не мешает: тестовая
фабрика создаёт Settings напрямую.
</details>

## 🎓 Итоги

- pydantic-settings: приоритеты «конструктор → env → .env → дефолт», валидация
  на старте, весь конфиг в одном классе. 127.0.0.1 — выстраданная константа.
- Слои: роутеры (HTTP) → сервисы (логика) → клиенты (IO); индексация — скрипт.
- Чанкинг markdown: по `##`-секциям, длинное — по абзацам с перекрытием;
  заголовки секций = бесплатные контекстные заголовки.
- Индексация идемпотентна благодаря uuid5(source#chunk_index); живой прогон:
  12 документов → 61 чанк → 61 точка.

**Дальше — урок 8.3:** первый эндпоинт — `/api/search`: эмбеддинг запроса,
ANN в Qdrant, двухступенчатый поиск с reranker и честное сравнение шкал.